# CÂU 6: PHÂN TÍCH TẦN SUẤT MUA HÀNG THEO NHÓM TUỔI

**1. Yêu cầu bài toán:**
Tính số đơn hàng trung bình trên mỗi khách hàng theo từng nhóm tuổi (age_group) để tìm ra nhóm có chỉ số cao nhất. Lưu ý: Chỉ xét các khách hàng có `age_group` khác null.

**2. Phương pháp tiếp cận & Suy luận:**
* **Bước 1:** Tải dữ liệu từ 2 file `customers.csv` và `orders.csv`.
* **Bước 2:** Lọc bỏ các khách hàng không có thông tin nhóm tuổi (`age_group` bị null).
* **Bước 3:** Đếm tổng số lượng khách hàng cho từng nhóm tuổi.
* **Bước 4:** Nối (Join) bảng đơn hàng với bảng khách hàng để biết mỗi đơn thuộc về nhóm tuổi nào. Sau đó đếm tổng số đơn hàng của từng nhóm.
* **Bước 5:** Lấy Tổng số đơn (Bước 4) chia cho Tổng số khách hàng (Bước 3) để ra chỉ số trung bình và tìm nhóm cao nhất.

**3. Lý do chọn phương pháp:**
* **Tính toán tách biệt tử số và mẫu số:** Ta không thể chỉ nối bảng rồi dùng hàm trung bình (`mean()`) thông thường. Việc tính riêng "số lượng khách hàng" (`nunique()`) từ bảng `customers` đảm bảo mẫu số bao gồm cả những người đã đăng ký tài khoản nhưng chưa từng mua hàng, phản ánh chính xác nhất sức mua thực tế của toàn bộ nhóm tuổi đó theo đúng định nghĩa bài toán.
* **Sử dụng `dropna()`:** Cách nhanh và sạch nhất trong Pandas để loại bỏ các dòng bị khuyết thiếu dữ liệu trước khi đưa vào phân tích.

In [1]:
# Import thư viện
import pandas as pd
from google.colab import drive

# Kết nối với Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


**Thực hiện Bước 1, 2 & 3: Tiền xử lý bảng Khách hàng**

Ta sẽ load file dữ liệu, loại bỏ những ai không khai báo tuổi, và chốt xem mỗi nhóm tuổi có chính xác bao nhiêu khách hàng.

In [2]:
# 1. Khai báo đường dẫn
path_cust = '/content/drive/MyDrive/datathon-2026-round-1/customers.csv'
path_orders = '/content/drive/MyDrive/datathon-2026-round-1/orders.csv'

# 2. Đọc dữ liệu
df_customers = pd.read_csv(path_cust)
df_orders = pd.read_csv(path_orders)

# Lọc bỏ các dòng mà age_group bị trống (null)
df_customers_valid = df_customers.dropna(subset=['age_group'])

# Đếm tổng số khách hàng duy nhất (mẫu số) cho từng nhóm tuổi
cust_count_per_age = df_customers_valid.groupby('age_group')['customer_id'].nunique()

print("Số lượng khách hàng theo từng nhóm tuổi:")
print(cust_count_per_age)

Số lượng khách hàng theo từng nhóm tuổi:
age_group
18-24    17039
25-34    36342
35-44    31920
45-54    23172
55+      13457
Name: customer_id, dtype: int64


**Thực hiện Bước 4 & 5: Đếm số đơn hàng và Tính tỷ lệ**

Tiếp theo, ta nối 2 bảng lại để đếm xem mỗi nhóm tuổi tạo ra bao nhiêu đơn hàng, sau đó làm phép chia để ra con số trung bình cuối cùng.

In [3]:
# Nối bảng đơn hàng với bảng khách hàng hợp lệ
df_merged = pd.merge(df_orders, df_customers_valid, on='customer_id', how='inner')

# Đếm tổng số đơn hàng (tử số) cho từng nhóm tuổi
order_count_per_age = df_merged.groupby('age_group')['order_id'].count()

# Tính số đơn trung bình trên mỗi khách hàng và sắp xếp giảm dần
avg_orders = (order_count_per_age / cust_count_per_age).sort_values(ascending=False)

print("Số đơn hàng trung bình trên mỗi khách hàng:")
print(avg_orders)

# Xác định nhóm cao nhất
best_age_group = avg_orders.idxmax()
max_avg = avg_orders.max()

print(f"\n--> Nhóm tuổi có số đơn hàng trung bình cao nhất là: '{best_age_group}' ({max_avg:.2f} đơn/khách)")

Số đơn hàng trung bình trên mỗi khách hàng:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64

--> Nhóm tuổi có số đơn hàng trung bình cao nhất là: '55+' (5.41 đơn/khách)


**KẾT LUẬN CUỐI CÙNG:**
Dựa trên kết quả tính toán sau khi kết nối bảng dữ liệu khách hàng và đơn hàng, nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất là **55+** (với 5.4068 đơn/khách).

Đối chiếu với các phương án của đề bài:

A) 55+

B) 25-34

C) 35-44

D) 45-54

**=> Đáp án được chọn là: A**